# Integrated Notebook — Mechanisms 1–5 (CSP → Cookies → Security Headers → Input Validation/Reflection → DOM XSS)

This notebook runs **Mechanisms 1–5** in one place to save time.

- **Passive by default** for public sites.
- **DVWA**: supports **auto-login + security level** (optional).
- You can add any targets in the `TARGETS` list.

> Tip: For DVWA targets, set `DVWA_BASE`, `DVWA_USERNAME`, `DVWA_PASSWORD`.


In [ ]:
# =========================
# CONFIG (edit this only)
# =========================

TARGETS = [
    # DVWA (lab)
    "http://192.168.88.129/dvwa/vulnerabilities/xss_r/",

    # Public sites (passive only)
    "https://example.com/",
    "https://owasp.org/",
    "https://github.com/login",
]

# DVWA optional login (leave as None if not using DVWA)
DVWA_BASE = "http://192.168.88.129/dvwa/"
DVWA_USERNAME = "admin"
DVWA_PASSWORD = "password"
DVWA_SECURITY_LEVEL = "low"   # low / medium / high / impossible

# Safety: keep active testing OFF for public sites
PASSIVE_ONLY_FOR_PUBLIC = True


## Mechanism 1 — CSP Check (Passive)

In [ ]:
"""
MECHANISM #3 — Content Security Policy (CSP) Check (PASSIVE)
FULL CODE (DVWA + Juice Shop) with:
✅ Clean output format (same as your sample)
✅ Correct CSP logic (header + report-only + meta tag)
✅ Handles SPA "#/" routes (removes # fragment)
✅ Handles 5xx (503 etc.) as INCONCLUSIVE (not NO_CSP)
✅ Uses "browser-like headers" so Juice Shop demo is less likely to return 503
   (Edge works because it sends these types of headers)

Passive scanner means:
- We DO NOT inject payloads.
- We only read headers and HTML.

You can use:
- Juice Shop demo:  https://demo.owasp-juice.shop/#/
- OR local Juice Shop: http://localhost:3000/
"""

import requests
from typing import Dict, Optional, List
from urllib.parse import urldefrag
from bs4 import BeautifulSoup


# =========================================================
# Browser-like headers (helps when sites block bots/requests)
# =========================================================
BROWSER_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Connection": "close",
    # Some CDNs behave better when a Referer exists (optional)
    "Referer": "https://demo.owasp-juice.shop/",
}


# =========================================================
# DVWA HELPERS (Login + set security level)
# =========================================================

def extract_csrf_token(html_text: str, token_name: str = "user_token") -> Optional[str]:
    """
    DVWA uses a CSRF token called 'user_token' in hidden input fields.
    This reads it from HTML.
    """
    soup = BeautifulSoup(html_text, "html.parser")
    inp = soup.find("input", {"name": token_name})
    return inp.get("value") if inp and inp.get("value") else None


def dvwa_login(login_url: str, username: str, password: str) -> requests.Session:
    """
    Logs into DVWA and returns an authenticated session.
    (Session stores cookies, so next requests stay logged in.)
    """
    s = requests.Session()

    # Step 1: GET login page (to capture CSRF token if present)
    r1 = s.get(login_url, timeout=10, headers=BROWSER_HEADERS)
    token = extract_csrf_token(r1.text, "user_token")

    # Step 2: POST login data
    data = {"username": username, "password": password, "Login": "Login"}
    if token:
        data["user_token"] = token

    s.post(login_url, data=data, allow_redirects=True, timeout=10, headers=BROWSER_HEADERS)
    return s


def dvwa_set_security(session: requests.Session, security_url: str, level: str = "high") -> None:
    """
    Sets DVWA security level to: low / medium / high / impossible.
    """
    r = session.get(security_url, timeout=10, headers=BROWSER_HEADERS)
    token = extract_csrf_token(r.text, "user_token")

    data = {"security": level, "seclev_submit": "Submit"}
    if token:
        data["user_token"] = token

    session.post(security_url, data=data, allow_redirects=True, timeout=10, headers=BROWSER_HEADERS)


# =========================================================
# CSP PARSING + ANALYSIS (Correct)
# =========================================================

def parse_csp_directives(csp_string: str) -> Dict[str, List[str]]:
    """
    Converts CSP text into dict of directives.

    Example:
      "default-src 'none'; script-src 'self' https://cdn.com"
    becomes:
      {"default-src": ["'none'"], "script-src": ["'self'", "https://cdn.com"]}

    IMPORTANT:
    - Directive names are lowercased so checks always work.
    """
    directives: Dict[str, List[str]] = {}
    if not csp_string:
        return directives

    parts = [p.strip() for p in csp_string.split(";") if p.strip()]
    for part in parts:
        tokens = part.split()
        if not tokens:
            continue
        name = tokens[0].lower()
        values = tokens[1:] if len(tokens) > 1 else []
        directives[name] = values

    return directives


def extract_csp_from_meta(html_text: str) -> Optional[str]:
    """
    Some websites use CSP via HTML meta tag:
      <meta http-equiv="Content-Security-Policy" content="...">

    This extracts that content if present.
    """
    soup = BeautifulSoup(html_text, "html.parser")
    meta = soup.find("meta", attrs={"http-equiv": lambda v: v and v.lower() == "content-security-policy"})
    if meta and meta.get("content"):
        return meta["content"].strip()
    return None


def analyze_csp_vulnerabilities(directives: Dict[str, List[str]]) -> List[str]:
    """
    Finds common weak CSP patterns.
    """
    vulns: List[str] = []

    default_src = directives.get("default-src", [])
    script_src = directives.get("script-src", default_src)
    style_src = directives.get("style-src", default_src)

    # Weak script settings
    if "'unsafe-inline'" in script_src:
        vulns.append("'unsafe-inline' in script-src - allows inline <script> tags and event handlers")

    if "'unsafe-eval'" in script_src:
        vulns.append("'unsafe-eval' in script-src - allows eval() and Function() constructor")

    if "*" in script_src:
        vulns.append("Wildcard (*) in script-src - allows scripts from ANY domain")

    if "data:" in script_src:
        vulns.append("'data:' in script-src - allows data: URI scripts (can increase XSS risk)")

    # Insecure sources
    for value in script_src:
        if value.startswith("http://"):
            vulns.append(f"HTTP (non-HTTPS) source allowed in script-src: {value} - man-in-the-middle risk")
        if value.startswith("https://*.") or value.startswith("*."):
            vulns.append(f"Wildcard subdomain in script-src: {value} - broader attack surface")

    # base-uri protects against <base> tag injection
    if "base-uri" not in directives:
        vulns.append("Missing 'base-uri' directive - base tag injection not restricted")

    # object-src FIX:
    # If object-src missing, it falls back to default-src.
    # If default-src has 'none', objects are already blocked -> not a vulnerability.
    if "object-src" not in directives:
        if "'none'" not in default_src:
            vulns.append("Missing 'object-src' directive - plugins/objects not explicitly restricted")
    else:
        if "'none'" not in directives.get("object-src", []):
            vulns.append("object-src is not 'none' - plugins/objects may be allowed")

    # form-action limits where forms can submit
    if "form-action" not in directives:
        vulns.append("Missing 'form-action' directive - form submission destination not restricted")

    # style-src unsafe-inline (lower risk than scripts)
    if "'unsafe-inline'" in style_src:
        vulns.append("'unsafe-inline' in style-src - inline CSS allowed (lower risk than scripts)")

    return vulns


def generate_csp_recommendations(directives: Dict[str, List[str]]) -> List[str]:
    """
    Generates recommendations based on the CSP configuration.
    """
    if not directives:
        return [
            "Implement Content-Security-Policy header (preferred) or CSP meta tag",
            "Start with CSP Report-Only mode to test",
            "Use 'default-src' and 'script-src' directives",
            "Block 'unsafe-inline' and 'unsafe-eval'"
        ]

    recs: List[str] = []
    default_src = directives.get("default-src", [])
    script_src = directives.get("script-src", default_src)

    if "'unsafe-inline'" in script_src:
        recs.append("Remove 'unsafe-inline' and use nonces/hashes for required inline scripts")

    if "'unsafe-eval'" in script_src:
        recs.append("Remove 'unsafe-eval' - avoid eval()/Function() patterns")

    if "*" in script_src or "data:" in script_src:
        recs.append("Restrict script-src to trusted domains only (avoid '*' and data:)")

    if "base-uri" not in directives:
        recs.append("Add: base-uri 'self'")

    if "object-src" not in directives and "'none'" not in default_src:
        recs.append("Add: object-src 'none'")
    elif "object-src" in directives and "'none'" not in directives.get("object-src", []):
        recs.append("Set: object-src 'none'")

    if "form-action" not in directives:
        recs.append("Add: form-action 'self'")

    if "default-src" not in directives:
        recs.append("Add: default-src 'self' (or 'none' for strict baseline)")

    if "upgrade-insecure-requests" not in directives:
        recs.append("Consider adding: upgrade-insecure-requests (forces HTTPS where possible)")

    if not recs:
        recs.append("CSP looks strong - keep it maintained and tested")

    return recs


# =========================================================
# MECHANISM #3: CSP CHECK (PASSIVE)
# =========================================================

def check_content_security_policy(session: requests.Session, target_url: str) -> Dict:
    """
    PASSIVE CSP check:
    - Removes #fragment (important for SPAs like Juice Shop)
    - Uses browser-like headers (reduce 503)
    - Reads CSP headers or CSP meta tag
    - If status is 5xx, returns INCONCLUSIVE
    """
    try:
        clean_url, _ = urldefrag(target_url)

        response = session.get(
            clean_url,
            allow_redirects=True,
            timeout=12,
            headers=BROWSER_HEADERS
        )

        # If server error (503 etc.), do NOT conclude NO_CSP
        if 500 <= response.status_code <= 599:
            return {
                "final_url": response.url,
                "status_code": response.status_code,
                "csp_present": "INCONCLUSIVE",
                "csp_type": None,
                "csp_header": None,
                "csp_report_only": None,
                "csp_meta": None,
                "verdict": "INCONCLUSIVE_SERVER_ERROR",
                "risk_level": "UNKNOWN",
                "evidence": f"Server returned {response.status_code}. Cannot reliably assess CSP from an error response.",
                "directives": {},
                "vulnerabilities": [],
                "recommendations": [
                    "Retry against a healthy instance (prefer local Juice Shop).",
                    "Scan base URL without # fragment (example: https://demo.owasp-juice.shop/)."
                ]
            }

        csp_header = response.headers.get("Content-Security-Policy")
        csp_report_only = response.headers.get("Content-Security-Policy-Report-Only")

        # If no header, check meta CSP in HTML
        csp_meta = None
        if not csp_header and not csp_report_only:
            csp_meta = extract_csp_from_meta(response.text)

        # Still nothing -> CSP missing
        if not csp_header and not csp_report_only and not csp_meta:
            return {
                "final_url": response.url,
                "status_code": response.status_code,
                "csp_present": "NO",
                "csp_type": None,
                "csp_header": None,
                "csp_report_only": None,
                "csp_meta": None,
                "verdict": "NO_CSP",
                "risk_level": "HIGH",
                "evidence": "No CSP header or CSP meta tag found",
                "directives": {},
                "vulnerabilities": ["Missing CSP - scripts are less restricted by browser policy"],
                "recommendations": [
                    "Implement Content-Security-Policy header",
                    "Start with CSP Report-Only mode to test",
                    "Use 'default-src' and 'script-src' directives",
                    "Block 'unsafe-inline' and 'unsafe-eval'"
                ]
            }

        # Choose enforcing header > report-only header > meta
        if csp_header:
            active = csp_header
            csp_type = "Enforcing (Header)"
        elif csp_report_only:
            active = csp_report_only
            csp_type = "Report-Only (Header)"
        else:
            active = csp_meta
            csp_type = "Enforcing (Meta)"

        directives = parse_csp_directives(active)
        vulnerabilities = analyze_csp_vulnerabilities(directives)
        recommendations = generate_csp_recommendations(directives)

        if csp_type.startswith("Report-Only"):
            verdict = "CSP_REPORT_ONLY"
            risk_level = "MEDIUM"
        elif vulnerabilities:
            verdict = "CSP_WEAK"
            risk_level = "MEDIUM" if len(vulnerabilities) <= 2 else "HIGH"
        else:
            verdict = "CSP_STRONG"
            risk_level = "LOW"

        return {
            "final_url": response.url,
            "status_code": response.status_code,
            "csp_present": "YES",
            "csp_type": csp_type,
            "csp_header": csp_header,
            "csp_report_only": csp_report_only,
            "csp_meta": csp_meta,
            "verdict": verdict,
            "risk_level": risk_level,
            "evidence": f"{csp_type} CSP found with {len(directives)} directives",
            "directives": directives,
            "vulnerabilities": vulnerabilities,
            "recommendations": recommendations
        }

    except Exception as e:
        return {
            "final_url": target_url,
            "status_code": None,
            "csp_present": "ERROR",
            "csp_type": None,
            "csp_header": None,
            "csp_report_only": None,
            "csp_meta": None,
            "verdict": "ERROR",
            "risk_level": "UNKNOWN",
            "evidence": f"Error during CSP check: {str(e)}",
            "directives": {},
            "vulnerabilities": [],
            "recommendations": []
        }


def print_csp_results(result: Dict) -> None:
    """
    Clean output format (same style you asked).
    """
    print("\n" + "=" * 72)
    print("MECHANISM #3: CONTENT SECURITY POLICY (CSP) CHECK")
    print("=" * 72)
    print(f"Final URL:          {result['final_url']}")
    print(f"Status Code:        {result['status_code']}")
    print(f"CSP Present:        {result['csp_present']}")
    print("-" * 72)
    print("-" * 72)

    if result.get("directives"):
        print("CSP Directives Found:")
        for directive, values in result["directives"].items():
            values_str = " ".join(values) if values else "(no values)"
            print(f"  • {directive:20s} : {values_str}")
    else:
        print("CSP Directives:     None")

    print("-" * 72)
    print(f"Verdict:            {result['verdict']}")
    print(f"Risk Level:         {result['risk_level']}")
    print(f"Evidence:           {result['evidence']}")

    if result.get("vulnerabilities"):
        print("-" * 72)
        print("⚠️  Vulnerabilities Found:")
        for i, vuln in enumerate(result["vulnerabilities"], 1):
            print(f"  {i}. {vuln}")

    if result.get("recommendations"):
        print("-" * 72)
        print("💡 Recommendations:")
        for i, rec in enumerate(result["recommendations"], 1):
            print(f"  {i}. {rec}")

    print("=" * 72)


# =========================================================
# MAIN: DVWA + Juice Shop
# =========================================================

## Mechanism 2 — Cookie & Session Hardening (includes session enforcement)

In [ ]:
import re
from urllib.parse import urlparse
from email.utils import parsedate_to_datetime

def mech2_cookie_session_hardening(session, target_url: str, timeout: int = 15, verify_tls: bool = True):
    """
    Cookie & Session Hardening (Clean)
    - Uses the *existing* session (so you don't handle cookies twice)
    - Fetches target_url
    - Checks:
        HttpOnly / Secure / SameSite / Expiry
    - Session enforcement:
        If redirected to login.php, prints that the endpoint is protected
    Returns a dict report.
    """

    print("\n" + "="*70)
    print("MECHANISM #2 — COOKIE & SESSION HARDENING")
    print("="*70)

    r = session.get(target_url, allow_redirects=True, timeout=timeout, verify=verify_tls)
    final_url = r.url
    scheme = urlparse(final_url).scheme

    print(f"\nTarget URL  : {target_url}")
    print(f"Final URL   : {final_url}")
    print(f"HTTP Status : {r.status_code}")

    # Session enforcement check
    redirected_to_login = "login.php" in final_url.lower()
    print("\n[Session Enforcement]")
    if redirected_to_login:
        print("✅ Redirected to login page → endpoint requires authentication (protected).")
    else:
        print("ℹ️ Not redirected to login → page may be public OR session already authenticated.")

    # Collect Set-Cookie headers
    if hasattr(r.raw.headers, "get_all"):
        cookies_raw = r.raw.headers.get_all("Set-Cookie") or []
    else:
        sc = r.headers.get("Set-Cookie")
        cookies_raw = [sc] if sc else []

    if not cookies_raw:
        print("\n[Cookie Flags]")
        print("⚠️ No Set-Cookie headers observed on this response.")
        return {
            "target_url": target_url,
            "final_url": final_url,
            "status": r.status_code,
            "redirected_to_login": redirected_to_login,
            "total_cookies": 0,
            "high_risk_issues": 0,
            "cookies": []
        }

    print("\n" + "-"*70)
    print("COOKIE ANALYSIS")
    print("-"*70)

    results = []
    total = 0
    high_risk = 0

    auth_keywords = ["sess", "session", "phpsessid", "auth", "token", "jwt"]

    for cookie in cookies_raw:
        total += 1
        first = cookie.split(";", 1)[0]
        name = first.split("=")[0].strip()

        httponly = "httponly" in cookie.lower()
        secure = "secure" in cookie.lower()

        samesite_match = re.search(r"SameSite=([^;]+)", cookie, re.I)
        samesite = samesite_match.group(1).strip() if samesite_match else None

        expires_m = re.search(r"Expires=([^;]+)", cookie, re.I)
        maxage_m = re.search(r"Max-Age=([^;]+)", cookie, re.I)

        is_auth = any(k in name.lower() for k in auth_keywords)

        print(f"\nCookie Name : {name}")
        print(f"Type        : {'AUTH/SESSION' if is_auth else 'Other'}")
        print(f"HttpOnly    : {'PASS' if httponly else 'WARNING'}")
        print(f"Secure      : {'PASS' if secure else ('INFO (HTTP site)' if scheme=='http' else 'WARNING')}")
        print(f"SameSite    : {samesite if samesite else 'WARNING'}")

        expiry = "Session cookie"
        if expires_m:
            exp_str = expires_m.group(1).strip()
            try:
                exp_dt = parsedate_to_datetime(exp_str)
                expiry = f"Expires: {exp_str} (parsed: {exp_dt})"
            except Exception:
                expiry = f"Expires: {exp_str}"
        elif maxage_m:
            expiry = f"Max-Age: {maxage_m.group(1).strip()}"

        print(f"Expires     : {expiry}")

        # high risk: session cookie missing HttpOnly OR (https and missing Secure)
        issues = []
        if is_auth and not httponly:
            issues.append("Missing HttpOnly")
            high_risk += 1
        if is_auth and scheme == "https" and not secure:
            issues.append("Missing Secure (HTTPS)")
            high_risk += 1

        results.append({
            "name": name,
            "is_auth": is_auth,
            "httponly": httponly,
            "secure": secure,
            "samesite": samesite,
            "expiry": expiry,
            "issues": issues
        })

    print("\n" + "-"*70)
    print("SUMMARY")
    print("-"*70)
    print(f"Total Cookies Observed : {total}")
    print(f"High-Risk Issues       : {high_risk}")
    print(f"Redirected to login    : {redirected_to_login}")

    if high_risk == 0:
        print("Overall Assessment     : PASS")
    else:
        print("Overall Assessment     : WARNING")

    return {
        "target_url": target_url,
        "final_url": final_url,
        "status": r.status_code,
        "redirected_to_login": redirected_to_login,
        "total_cookies": total,
        "high_risk_issues": high_risk,
        "cookies": results
    }


## Mechanism 3 — Security Headers Audit (Passive)

In [ ]:
import requests
from urllib.parse import urlparse

# ============================================================
# MECHANISM #3 — SECURITY HEADERS AUDIT (PASSIVE) [UPDATED]
# ============================================================
# ✅ Beginner-friendly comments
# ✅ Works on DVWA + public websites
# ✅ Fix #1: also checks "Content-Security-Policy-Report-Only"
# ✅ Fix #2: Referrer-Policy check uses "contains" instead of "startswith"
#
# What are security headers?
# --------------------------
# They are special HTTP response headers that tell the browser
# to behave more safely and reduce common attacks.
#
# This is PASSIVE scanning:
# ✅ We only request the page and read the response headers
# ❌ We do not attack / inject payloads
# ============================================================


def mechanism3_security_headers_audit(target_url: str) -> dict:
    """
    Fetches a URL and checks common security headers.
    Returns a dictionary result (easy to integrate later).
    """

    # Browser-like headers help reduce bot blocks on some websites
    browser_headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "close",
    }

    # ----------------------------
    # STEP 1: Make request
    # ----------------------------
    session = requests.Session()
    try:
        r = session.get(target_url, allow_redirects=True, timeout=15, headers=browser_headers)
    except Exception as e:
        return {
            "target_url": target_url,
            "final_url": target_url,
            "status_code": None,
            "error": str(e),
            "verdict": "ERROR",
            "risk_level": "UNKNOWN",
            "checks": {},
        }

    final_url = r.url
    status = r.status_code
    scheme = urlparse(final_url).scheme.lower()
    headers = r.headers

    # ----------------------------
    # STEP 2: Define checks
    # ----------------------------
    # For each header we store:
    # - why it matters
    # - recommended values (if simple)
    header_rules = {
        "Strict-Transport-Security": {
            "why": "Forces HTTPS in browser (prevents downgrade). Only meaningful on HTTPS sites.",
            "only_https": True,
        },
        "X-Content-Type-Options": {
            "why": "Stops MIME sniffing (browser guessing file types).",
            "recommended": ["nosniff"],
        },
        "X-Frame-Options": {
            "why": "Helps prevent clickjacking (embedding page inside attacker iframe).",
            "recommended": ["deny", "sameorigin"],
        },
        "Referrer-Policy": {
            "why": "Controls how much referrer information leaks to other sites.",
            # NOTE: websites may return multiple values separated by comma
            "recommended": ["no-referrer", "same-origin", "strict-origin", "strict-origin-when-cross-origin"],
        },
        "Permissions-Policy": {
            "why": "Limits browser features like camera, mic, geolocation etc.",
            "recommended": None,  # too many valid formats; we only check presence
        },
        "Cross-Origin-Opener-Policy": {
            "why": "Helps isolate browsing context and reduce cross-origin attacks.",
            "recommended": ["same-origin", "same-origin-allow-popups"],
        },
        "Cross-Origin-Embedder-Policy": {
            "why": "Strong isolation header; blocks unsafe cross-origin embedded resources.",
            "recommended": ["require-corp", "credentialless"],
        },
        "Cross-Origin-Resource-Policy": {
            "why": "Controls which origins can load your resources.",
            "recommended": ["same-origin", "same-site", "cross-origin"],
        },

        # CSP is Mechanism #1 in your project (deep analysis there).
        # Here we only check if it's present.
        "Content-Security-Policy": {
            "why": "CSP reduces XSS impact by restricting scripts/resources. (Deep check is Mechanism #1.)",
            "recommended": None,
        },

        # ✅ NEW: Report-only CSP (very common on Google and others)
        "Content-Security-Policy-Report-Only": {
            "why": "CSP in monitoring mode (reports violations but does not block). Still valuable.",
            "recommended": None,
        },
    }

    # ----------------------------
    # STEP 3: Importance levels
    # ----------------------------
    # High importance = core basic protection headers most apps should have
    high_importance = {"X-Content-Type-Options", "X-Frame-Options", "Referrer-Policy"}

    # Medium importance = good modern headers
    medium_importance = {
        "Strict-Transport-Security",
        "Permissions-Policy",
        "Cross-Origin-Opener-Policy",
        "Cross-Origin-Embedder-Policy",
        "Cross-Origin-Resource-Policy",
        "Content-Security-Policy",
        "Content-Security-Policy-Report-Only",
    }

    checks = {}
    missing_high = 0
    missing_med = 0

    # ----------------------------
    # STEP 4: Evaluate headers
    # ----------------------------
    for hname, rule in header_rules.items():
        raw_val = headers.get(hname)

        # Special case: HSTS only relevant on HTTPS sites
        if rule.get("only_https") and scheme != "https":
            checks[hname] = {
                "present": False if raw_val is None else True,
                "value": raw_val,
                "status": "INFO",
                "note": "Meaningful only on HTTPS (this site is HTTP).",
                "why": rule["why"],
            }
            continue

        # Missing header
        if raw_val is None:
            if hname in high_importance:
                missing_high += 1
            elif hname in medium_importance:
                missing_med += 1

            checks[hname] = {
                "present": False,
                "value": None,
                "status": "MISSING",
                "note": "Header not found.",
                "why": rule["why"],
            }
            continue

        # Present header
        recommended = rule.get("recommended")

        # If no strict recommended list: just confirm presence
        if recommended is None:
            checks[hname] = {
                "present": True,
                "value": raw_val,
                "status": "PRESENT",
                "note": "Header present (value not strictly validated here).",
                "why": rule["why"],
            }
            continue

        # If recommended list exists, validate (case-insensitive)
        val_low = raw_val.strip().lower()

        # ✅ FIX: Use "contains" check, not only "startswith"
        # because some sites return multiple referrer policies separated by commas.
        if any(x in val_low for x in recommended):
            checks[hname] = {
                "present": True,
                "value": raw_val,
                "status": "PASS",
                "note": "Looks good.",
                "why": rule["why"],
            }
        else:
            checks[hname] = {
                "present": True,
                "value": raw_val,
                "status": "WEAK",
                "note": f"Present but not matching recommended values: {recommended}",
                "why": rule["why"],
            }
            # Weak values count as medium issues
            missing_med += 1

    # ----------------------------
    # STEP 5: Overall verdict
    # ----------------------------
    if status is not None and 500 <= status <= 599:
        verdict = "INCONCLUSIVE_SERVER_ERROR"
        risk = "UNKNOWN"
    elif missing_high >= 2:
        verdict = "WEAK_HEADERS"
        risk = "HIGH"
    elif missing_high == 1 or missing_med >= 3:
        verdict = "PARTIAL_HEADERS"
        risk = "MEDIUM"
    else:
        verdict = "GOOD_HEADERS"
        risk = "LOW"

    return {
        "target_url": target_url,
        "final_url": final_url,
        "status_code": status,
        "verdict": verdict,
        "risk_level": risk,
        "checks": checks,
        "missing_high": missing_high,
        "missing_med": missing_med,
        "scheme": scheme,
    }


def print_mechanism3_report(result: dict) -> None:
    """
    Prints a clean report output.
    """

    print("\n" + "=" * 72)
    print("MECHANISM #3: SECURITY HEADERS AUDIT (PASSIVE)")
    print("=" * 72)

    if result.get("error"):
        print(f"Target URL : {result['target_url']}")
        print(f"Error      : {result['error']}")
        print("=" * 72)
        return

    print(f"Target URL : {result['target_url']}")
    print(f"Final URL  : {result['final_url']}")
    print(f"Status     : {result['status_code']}")
    print("-" * 72)

    for hname, info in result["checks"].items():
        status = info["status"]
        value = info["value"]

        # If very long header value, show only first part (clean output)
        if isinstance(value, str) and len(value) > 120:
            value_disp = value[:120] + "..."
        else:
            value_disp = value

        print(f"{hname:35s}: {status}")
        if value_disp is not None:
            print(f"  Value : {value_disp}")
        print(f"  Why   : {info['why']}")
        print(f"  Note  : {info['note']}")
        print("-" * 72)

    print(f"Verdict   : {result['verdict']}")
    print(f"Risk      : {result['risk_level']}")
    print(f"Notes     : Missing HIGH={result['missing_high']}, Missing/Weak MED={result['missing_med']}")
    print("=" * 72)


# ============================================================
# TEST SECTION — run on multiple URLs
# ============================================================

## Mechanism 4 — Input Validation & Reflection (Passive by default)

In [ ]:
"""
MECHANISM #4 — INPUT VALIDATION (PASSIVE + ACTIVE)

Goal:
1) PASSIVE: Observe what the website *claims* to validate (forms, HTML attributes, headers).
2) ACTIVE: Send safe test inputs to see how the server *actually* validates/rejects inputs.

Why both?
- Passive shows "what is implemented on client side" (browser-side checks).
- Active confirms "real server-side validation", which matters for security.
"""

import html
import time
import requests
from typing import Dict, Optional, List, Tuple, Any
from bs4 import BeautifulSoup
from urllib.parse import urljoin


# ============================================================
# (A) CSRF TOKEN EXTRACTORS
# ============================================================
"""
Some websites use CSRF tokens to prevent unauthorized form submissions.
DVWA uses a token named "user_token".
Other websites may use: csrf_token, _token, etc.

We extract CSRF values so our session can submit forms correctly.
"""

def extract_csrf_token(html_text: str, token_name: str = "user_token") -> Optional[str]:
    """
    Extract one CSRF token value from a page by searching:
      <input name="TOKEN_NAME" value="...">

    Returns:
      token value (string) if found else None
    """
    soup = BeautifulSoup(html_text, "html.parser")
    inp = soup.find("input", {"name": token_name})
    if inp and inp.get("value"):
        return inp["value"]
    return None


def extract_any_csrf_token(html_text: str, token_names: List[str]) -> Tuple[Optional[str], Optional[str]]:
    """
    Try multiple common CSRF token field names.

    Returns:
      (token_field_name, token_value) if found else (None, None)
    """
    for name in token_names:
        val = extract_csrf_token(html_text, name)
        if val:
            return name, val
    return None, None


# ============================================================
# (B) GENERIC LOGIN FUNCTION
# ============================================================
"""
This helps us log into test web apps like DVWA using requests.Session().
Session keeps cookies so we stay logged in.
"""

def generic_login(
    login_url: str,
    username: str,
    password: str,
    username_field: str = "username",
    password_field: str = "password",
    submit_name: Optional[str] = None,
    submit_value: Optional[str] = None,
    csrf_token_names: Optional[List[str]] = None,
    success_keywords: Optional[List[str]] = None,
    timeout: int = 10,
) -> requests.Session:
    """
    Generic login:
    1) GET login page to collect CSRF token (if present)
    2) POST username/password (+ token)
    3) Return the session object
    """
    session = requests.Session()

    # Common token field names used by many apps
    token_names = csrf_token_names or ["user_token", "csrf_token", "_token", "csrf", "token"]

    # Keywords often present in logged-in pages
    keywords = success_keywords or ["logout", "sign out", "log out", "dashboard"]

    try:
        # Step 1: Open login page
        r1 = session.get(login_url, timeout=timeout)

        # Step 2: Try to extract CSRF token
        token_field, token_val = extract_any_csrf_token(r1.text, token_names)

        # Step 3: Prepare login data
        login_data = {
            username_field: username,
            password_field: password
        }

        # If token exists, include it
        if token_field and token_val:
            login_data[token_field] = token_val

        # Some forms require a submit field (DVWA uses Login=Login)
        if submit_name and submit_value:
            login_data[submit_name] = submit_value

        # Step 4: POST credentials
        r2 = session.post(login_url, data=login_data, allow_redirects=True, timeout=timeout)

        # Step 5: Very basic success check
        login_success = any(k in r2.text.lower() for k in keywords)
        if not login_success:
            print("⚠️ Warning: Login success not confirmed (keywords not found).")

        return session

    except Exception as e:
        print(f"❌ Login error: {e}")
        return session


# ============================================================
# (C) PASSIVE INPUT VALIDATION
# ============================================================
"""
PASSIVE means:
- We do NOT send payloads.
- We only observe:
  - How many forms exist?
  - Input constraints: required, maxlength, pattern, type=email, etc.
  - Security headers: CSP, etc.

This is safe for public websites too.
"""

def discover_forms(page_url: str, session: Optional[requests.Session] = None, timeout: int = 10) -> List[Dict[str, Any]]:
    """
    Extract all forms from a webpage.

    Returns a list like:
      [
        {
          "page_url": "...",
          "action_url": "...",
          "method": "GET/POST",
          "inputs": [
              {"name": "q", "type": "text", "required": True, ...}
          ]
        },
        ...
      ]
    """
    s = session or requests.Session()
    r = s.get(page_url, timeout=timeout, allow_redirects=True)

    soup = BeautifulSoup(r.text, "html.parser")
    forms = []

    for f in soup.find_all("form"):
        action = f.get("action") or ""
        method = (f.get("method") or "GET").upper()

        # action can be relative path, so join it with page URL
        target = urljoin(r.url, action)

        inputs = []
        for inp in f.find_all(["input", "textarea", "select"]):
            name = inp.get("name")
            if not name:
                continue

            inputs.append({
                "name": name,
                "type": (inp.get("type") or inp.name).lower(),
                "required": bool(inp.get("required")),
                "pattern": inp.get("pattern"),
                "maxlength": inp.get("maxlength"),
                "minlength": inp.get("minlength"),
                "min": inp.get("min"),
                "max": inp.get("max"),
                "autocomplete": inp.get("autocomplete"),
            })

        forms.append({
            "page_url": r.url,
            "action_url": target,
            "method": method,
            "inputs": inputs
        })

    return forms


def passive_input_validation_signals(page_url: str, session: Optional[requests.Session] = None, timeout: int = 10) -> Dict[str, Any]:
    """
    Collect passive signals:
    - security headers (CSP, XFO, X-CTO)
    - forms and client-side validation hints (pattern, maxlength, required)
    """
    s = session or requests.Session()
    r = s.get(page_url, timeout=timeout, allow_redirects=True)

    # Convert headers keys to lowercase for easy checks
    headers = {k.lower(): v for k, v in r.headers.items()}

    # Common security headers
    csp = headers.get("content-security-policy")
    xcto = headers.get("x-content-type-options")
    xfo = headers.get("x-frame-options")

    forms = discover_forms(page_url, session=s, timeout=timeout)

    # Collect input constraints (what browser may validate)
    client_constraints = []
    for idx, form in enumerate(forms, start=1):
        for inp in form["inputs"]:
            hints = []
            if inp.get("required"):
                hints.append("required")
            if inp.get("pattern"):
                hints.append(f"pattern={inp['pattern']}")
            if inp.get("maxlength"):
                hints.append(f"maxlength={inp['maxlength']}")
            if inp.get("minlength"):
                hints.append(f"minlength={inp['minlength']}")
            if inp.get("type") and inp["type"] not in ["text", "textarea"]:
                hints.append(f"type={inp['type']}")

            if hints:
                client_constraints.append({
                    "form_index": idx,
                    "action_url": form["action_url"],
                    "method": form["method"],
                    "field": inp["name"],
                    "hints": hints
                })

    return {
        "final_url": r.url,
        "status_code": r.status_code,
        "security_headers": {
            "content-security-policy": "present" if csp else "missing",
            "x-content-type-options": "present" if xcto else "missing",
            "x-frame-options": "present" if xfo else "missing",
        },
        "forms_found": len(forms),
        "client_side_constraints": client_constraints,
    }


# ============================================================
# (D) ACTIVE INPUT VALIDATION
# ============================================================
"""
ACTIVE means:
- We send test inputs (payloads) to check if server blocks/rejects.
- Use ONLY on authorized targets (DVWA, your lab, or permissioned websites).

We use a marker string to detect reflection/transformation.
This reduces false positives.
"""

def detect_validation_error(response: requests.Response) -> bool:
    """
    Detect if server indicates invalid input.

    We consider:
    - HTTP codes: 400/403/406/413/422/429 often represent blocking/rejection
    - error text keywords
    """
    error_indicators = [
        "invalid input", "validation error", "forbidden", "not allowed",
        "blocked", "rejected", "malicious", "suspicious", "bad request",
        "illegal characters", "request blocked", "access denied"
    ]

    status_code_errors = response.status_code in [400, 403, 406, 413, 422, 429]
    text_errors = any(ind in response.text.lower() for ind in error_indicators)

    return status_code_errors or text_errors


def reflection_classification(body: str, marker: str, raw_payload: str) -> str:
    """
    Check how the payload/marker appears in response.
    - RAW_REFLECTION: payload appears exactly as sent
    - HTML_ESCAPED_REFLECTION: appears encoded (&lt; &gt;)
    - MARKER_PRESENT_TRANSFORMED: marker appears but payload changed
    - NOT_REFLECTED: not visible in this response
    """
    if raw_payload in body:
        return "RAW_REFLECTION"

    encoded = html.escape(raw_payload, quote=True)
    if encoded in body:
        return "HTML_ESCAPED_REFLECTION"

    if marker in body:
        return "MARKER_PRESENT_TRANSFORMED"

    return "NOT_REFLECTED"


def check_input_validation_active(
    session: requests.Session,
    target_url: str,
    param_name: str,
    method: str = "GET",
    timeout: int = 10,
    delay_s: float = 0.0
) -> Dict[str, Any]:
    """
    Active test: send a few safe test payloads and analyze server behavior.
    """
    marker = "XSSMARK_9f3a2b1c"

    # Payloads are not for exploitation, only validation mapping
    test_payloads = {
        "benign": f"hello_{marker}",
        "html_tags": f"<b>{marker}</b>",
        "script_like": f"<script>{marker}</script>",
        "event_handler": f"<img src=x onerror={marker}>",
        "js_protocol": f"javascript:{marker}",
        "special_chars": f"<>\"'&;(){marker}",
        "long_input": ("A" * 300) + marker,
        "mixed_case_script": f"<ScRiPt>{marker}</sCrIpT>",
    }

    results = {}

    for payload_name, payload in test_payloads.items():
        if delay_s:
            time.sleep(delay_s)

        # Send GET/POST request based on chosen method
        if method.upper() == "POST":
            resp = session.post(target_url, data={param_name: payload}, allow_redirects=True, timeout=timeout)
        else:
            resp = session.get(target_url, params={param_name: payload}, allow_redirects=True, timeout=timeout)

        body = resp.text
        err = detect_validation_error(resp)

        # Heuristic:
        # - Accepted if server returns <400 and does not show error indicators
        accepted = (resp.status_code < 400) and (not err)

        results[payload_name] = {
            "payload": payload,
            "final_url": resp.url,
            "status_code": resp.status_code,
            "accepted": accepted,
            "error_detected": err,
            "reflection": reflection_classification(body, marker, payload),
            "response_length": len(body),
        }

    return analyze_validation_behavior(target_url, method, param_name, results)


def analyze_validation_behavior(target_url: str, method: str, param_name: str, results: Dict[str, Any]) -> Dict[str, Any]:
    """
    Convert raw results into a classification:
    - NO_VALIDATION
    - PATTERN_FILTERING
    - LENGTH_ONLY
    - PARTIAL
    """
    accepted = [k for k, v in results.items() if v["accepted"]]
    blocked = [k for k, v in results.items() if not v["accepted"]]

    benign_ok = "benign" in accepted
    script_blocked = "script_like" in blocked
    long_blocked = "long_input" in blocked

    # Filtering pattern hints
    patterns = []
    if results.get("script_like", {}).get("error_detected"):
        patterns.append("Script-like input triggers blocking")
    if results.get("event_handler", {}).get("error_detected"):
        patterns.append("Event-handler-like input triggers blocking")
    if results.get("mixed_case_script", {}).get("accepted") and not results.get("script_like", {}).get("accepted"):
        patterns.append("Case-sensitive filtering suspected")

    if not patterns:
        patterns = ["No clear filtering patterns detected"]

    # Classification rules (simple & explainable)
    if len(blocked) == 0:
        validation_type = "NO_VALIDATION"
        verdict = "NO_INPUT_VALIDATION"
        risk = "CRITICAL"
        evidence = "All payloads were accepted; no rejection signals detected."
        recs = [
            "Implement server-side allowlist validation (type, format, charset, length).",
            "Reject HTML/script-like inputs or encode them safely at output."
        ]
    elif not benign_ok:
        validation_type = "VERY_STRICT_OR_WRONG_PARAM"
        verdict = "BENIGN_REJECTED"
        risk = "INFO"
        evidence = "Even benign input rejected; check if correct parameter/endpoint is being tested."
        recs = [
            "Verify you selected the correct input parameter.",
            "If correct, validation may be strict allowlist."
        ]
    elif long_blocked and "script_like" in accepted:
        validation_type = "LENGTH_ONLY"
        verdict = "LENGTH_RESTRICTION_ONLY"
        risk = "HIGH"
        evidence = "Only length restriction is visible; script-like content still accepted."
        recs = [
            "Length checks alone are not enough for security.",
            "Add content/type validation + output encoding."
        ]
    elif script_blocked:
        validation_type = "PATTERN_FILTERING"
        verdict = "DANGEROUS_PATTERNS_BLOCKED"
        risk = "MEDIUM"
        evidence = "Script-like patterns blocked; likely blacklist/pattern filtering."
        recs = [
            "Blacklist filtering can be bypassed; move toward allowlist validation.",
            "Combine with context-aware output encoding."
        ]
    else:
        validation_type = "PARTIAL"
        verdict = "INCOMPLETE_FILTERING"
        risk = "MEDIUM-HIGH"
        evidence = "Some payloads blocked but behavior inconsistent."
        recs = [
            "Apply consistent allowlist validation server-side.",
            "Combine with output encoding to reduce XSS risk."
        ]

    return {
        "final_url": target_url,
        "method": method.upper(),
        "parameter": param_name,
        "test_results": results,
        "validation_type": validation_type,
        "verdict": verdict,
        "risk_level": risk,
        "evidence": evidence,
        "accepted_payloads": accepted,
        "blocked_payloads": blocked,
        "filtering_patterns": patterns,
        "recommendations": recs
    }


# ============================================================
# (E) PRINT RESULTS NICELY
# ============================================================
def print_passive_validation(passive: Dict[str, Any]) -> None:
    print("\n" + "=" * 72)
    print("MECHANISM #4A: PASSIVE INPUT VALIDATION SIGNALS")
    print("=" * 72)
    print(f"Final URL:      {passive['final_url']}")
    print(f"Status Code:    {passive['status_code']}")
    print(f"Forms Found:    {passive['forms_found']}")
    print("-" * 72)
    print("Security Headers:")
    for k, v in passive["security_headers"].items():
        print(f"  {k}: {v}")

    if passive["client_side_constraints"]:
        print("-" * 72)
        print("Client-side validation hints (HTML attributes):")
        for c in passive["client_side_constraints"][:20]:
            print(f"  • Form#{c['form_index']} {c['method']} {c['action_url']} | {c['field']} -> {', '.join(c['hints'])}")
        if len(passive["client_side_constraints"]) > 20:
            print(f"  ...and {len(passive['client_side_constraints']) - 20} more")

    print("=" * 72)


def print_active_validation(active: Dict[str, Any]) -> None:
    print("\n" + "=" * 72)
    print("MECHANISM #4B: ACTIVE INPUT VALIDATION TEST")
    print("=" * 72)
    print(f"Target URL:     {active['final_url']}")
    print(f"Method:         {active['method']}")
    print(f"Parameter:      {active['parameter']}")
    print("-" * 72)
    print(f"Validation Type:{active['validation_type']}")
    print(f"Verdict:        {active['verdict']}")
    print(f"Risk Level:     {active['risk_level']}")
    print(f"Evidence:       {active['evidence']}")
    print("-" * 72)
    print(f"Accepted: {len(active['accepted_payloads'])} | Blocked: {len(active['blocked_payloads'])}")
    print("Accepted payload keys:", ", ".join(active["accepted_payloads"]))
    print("Blocked  payload keys:", ", ".join(active["blocked_payloads"]))
    print("-" * 72)
    print("Filtering patterns:")
    for p in active["filtering_patterns"]:
        print(f"  • {p}")
    print("-" * 72)
    print("Recommendations:")
    for r in active["recommendations"]:
        print(f"  • {r}")
    print("=" * 72)


# ============================================================
# (F) RUNNER: TEST DVWA + OTHER WEBSITES
# ============================================================
"""
This part lets you test:
1) DVWA (passive + active)
2) Other websites (passive only unless you have permission)

Important concept:
- page_url = page where form exists
- action_url = endpoint where form submits
- param_name = name of input field
"""

def run_mechanism_4(
    session: Optional[requests.Session],
    page_url: str,
    action_url: Optional[str] = None,
    param_name: Optional[str] = None,
    method: Optional[str] = None,
    passive_only: bool = False
) -> Dict[str, Any]:
    """
    Runs:
    - passive_input_validation_signals() always
    - active testing only if passive_only=False and we have action_url+param_name
    """
    s = session or requests.Session()

    # PASSIVE analysis
    passive = passive_input_validation_signals(page_url, session=s)

    # If passive only, stop here
    if passive_only:
        return {"passive": passive, "active": None}

    # ACTIVE needs these
    if not action_url or not param_name or not method:
        return {
            "passive": passive,
            "active": {
                "validation_type": "ERROR",
                "verdict": "MISSING_TARGET_INFO",
                "risk_level": "UNKNOWN",
                "evidence": "Provide action_url + param_name + method to run active tests.",
                "accepted_payloads": [],
                "blocked_payloads": [],
                "filtering_patterns": [],
                "recommendations": ["Provide explicit endpoint and parameter."],
                "test_results": {}
            }
        }

    active = check_input_validation_active(
        session=s,
        target_url=action_url,
        param_name=param_name,
        method=method
    )

    return {"passive": passive, "active": active}


# ============================================================
# (G) EXAMPLE EXECUTION
# ============================================================
if __name__ == "__main__":

    # ---------------------------
    # Example 1: DVWA (authorized)
    # ---------------------------
    BASE = "http://192.168.88.129/dvwa"
    LOGIN_URL = f"{BASE}/login.php"
    SECURITY_URL = f"{BASE}/security.php"
    TARGET_PAGE = f"{BASE}/vulnerabilities/xss_r/"   # page with input field "name"

    # Login DVWA
    session_dvwa = generic_login(
        login_url=LOGIN_URL,
        username="admin",
        password="password",
        username_field="username",
        password_field="password",
        submit_name="Login",
        submit_value="Login"
    )

    # Set DVWA security level (optional)
    r = session_dvwa.get(SECURITY_URL, timeout=10)
    token = extract_csrf_token(r.text, "user_token")
    sec_data = {"security": "high", "seclev_submit": "Submit"}
    if token:
        sec_data["user_token"] = token
    session_dvwa.post(SECURITY_URL, data=sec_data, allow_redirects=True, timeout=10)

    # Run Mechanism #4 on DVWA
    dvwa_report = run_mechanism_4(
        session=session_dvwa,
        page_url=TARGET_PAGE,
        action_url=TARGET_PAGE,   # DVWA uses same URL for submission
        param_name="name",
        method="GET",
        passive_only=False
    )

    print_passive_validation(dvwa_report["passive"])
    print_active_validation(dvwa_report["active"])


    # ---------------------------------------------
    #

## Mechanism 5 — DOM XSS Sink/Source Scan (Static)

In [ ]:
import re
import requests
from urllib.parse import urljoin, urlparse

def mechanism5_dom_xss_scan(
    target_url: str,
    cookies: dict = None,
    timeout: int = 20,
    verify_tls: bool = True,
    max_external_js: int = 15,
    max_hits_per_rule: int = 3
):
    """
    MECHANISM #5 — DOM-based XSS Risk Scan (STATIC)
    - Downloads HTML
    - Scans inline <script> blocks
    - Fetches same-origin external JS (script src) and scans
    - Reports dangerous sinks + attacker-controlled sources with clear evidence
    """

    print("\n" + "="*78)
    print("MECHANISM #5 — DOM-BASED XSS SINK/SOURCE SCAN (STATIC)")
    print("="*78)

    s = requests.Session()
    if cookies:
        s.cookies.update(cookies)

    # Fetch HTML
    r = s.get(target_url, allow_redirects=True, timeout=timeout, verify=verify_tls)
    final_url = r.url
    status = r.status_code

    parsed = urlparse(final_url)
    base_origin = f"{parsed.scheme}://{parsed.netloc}"

    print(f"\nTarget URL : {target_url}")
    print(f"Final URL  : {final_url}")
    print(f"Status     : {status}")
    print(f"Origin     : {base_origin}")

    html_text = r.text

    # Inline scripts
    inline_scripts = re.findall(r"<script[^>]*>(.*?)</script>", html_text, flags=re.I | re.S)

    # External JS URLs (same-origin only for safety + speed)
    srcs = re.findall(r'<script[^>]+src=["\']([^"\']+)["\']', html_text, flags=re.I)
    external_js = []
    for src in srcs:
        full = urljoin(final_url, src)
        if urlparse(full).netloc == parsed.netloc:
            external_js.append(full)

    # Limit external JS fetches
    external_js = list(dict.fromkeys(external_js))[:max_external_js]

    print("\n--- Script inventory ---")
    print(f"Inline script blocks        : {len(inline_scripts)}")
    print(f"Same-origin external JS     : {len(external_js)} (scanning up to {max_external_js})")

    # DOM XSS sinks (dangerous write/execution points)
    SINK_RULES = {
        "innerHTML assignment": r"\.innerHTML\s*=",
        "outerHTML assignment": r"\.outerHTML\s*=",
        "insertAdjacentHTML": r"\.insertAdjacentHTML\s*\(",
        "document.write/ln": r"\bdocument\.write(?:ln)?\s*\(",
        "eval(": r"\beval\s*\(",
        "Function(": r"\bFunction\s*\(",
        "setTimeout('string')": r"\bsetTimeout\s*\(\s*['\"]",
        "setInterval('string')": r"\bsetInterval\s*\(\s*['\"]",
        "location =": r"\blocation\s*=\s*",
        "jQuery .html(": r"\.html\s*\(",
        "jQuery .append(": r"\.append\s*\(",
        "jQuery .prepend(": r"\.prepend\s*\(",
        "jQuery .before(": r"\.before\s*\(",
        "jQuery .after(": r"\.after\s*\(",
        "dangerous sink: srcdoc": r"\bsrcdoc\s*=",
    }

    # Attacker-controlled sources (untrusted inputs in the browser)
    SOURCE_RULES = {
        "location.href/search/hash": r"\blocation\.(href|search|hash)\b|\blocation\b",
        "document.URL/URI/baseURI": r"\bdocument\.(URL|documentURI|baseURI)\b",
        "document.referrer": r"\bdocument\.referrer\b",
        "document.cookie": r"\bdocument\.cookie\b",
        "localStorage": r"\blocalStorage\b",
        "sessionStorage": r"\bsessionStorage\b",
        "postMessage/onmessage": r"\bpostMessage\b|\bonmessage\b|\baddEventListener\s*\(\s*['\"]message['\"]",
    }

    def evidence_snippets(code: str, pattern: str, label: str, where: str):
        hits = []
        for m in re.finditer(pattern, code):
            start = max(0, m.start() - 60)
            end = min(len(code), m.end() + 60)
            snippet = code[start:end].replace("\n", " ").replace("\r", " ")
            hits.append({
                "rule": label,
                "where": where,
                "snippet": snippet
            })
            if len(hits) >= max_hits_per_rule:
                break
        return hits

    def scan_code(code: str, where: str):
        sink_hits = []
        src_hits = []
        for label, pat in SINK_RULES.items():
            sink_hits.extend(evidence_snippets(code, pat, label, where))
        for label, pat in SOURCE_RULES.items():
            src_hits.extend(evidence_snippets(code, pat, label, where))
        return sink_hits, src_hits

    all_sinks = []
    all_sources = []

    # Scan inline scripts
    for i, block in enumerate(inline_scripts, 1):
        sinks, srcs = scan_code(block, where=f"INLINE_SCRIPT_{i}")
        all_sinks.extend(sinks)
        all_sources.extend(srcs)

    # Fetch and scan external JS
    fetched_js = 0
    for js_url in external_js:
        try:
            jr = s.get(js_url, allow_redirects=True, timeout=timeout, verify=verify_tls)
            if jr.status_code == 200 and jr.text:
                fetched_js += 1
                sinks, srcs = scan_code(jr.text, where=f"JS:{js_url}")
                all_sinks.extend(sinks)
                all_sources.extend(srcs)
        except Exception:
            pass

    # Deduplicate hits roughly by (rule, where, snippet)
    def dedup(items):
        seen = set()
        out = []
        for it in items:
            key = (it["rule"], it["where"], it["snippet"])
            if key not in seen:
                seen.add(key)
                out.append(it)
        return out

    all_sinks = dedup(all_sinks)
    all_sources = dedup(all_sources)

    # Score heuristic: sinks weigh more
    score = min(30, len(all_sinks)*2 + len(all_sources))

    # Clear report
    print("\n--- Findings ---")
    print(f"External JS fetched         : {fetched_js}/{len(external_js)}")
    print(f"DOM XSS sinks found         : {len(all_sinks)}")
    print(f"Untrusted sources found     : {len(all_sources)}")
    print(f"Risk score (0–30 heuristic) : {score}")

    def print_hits(title, hits, limit=12):
        print("\n" + "-"*78)
        print(title)
        print("-"*78)
        if not hits:
            print("None found ✅")
            return
        for idx, h in enumerate(hits[:limit], 1):
            print(f"{idx}. Rule  : {h['rule']}")
            print(f"   Where : {h['where']}")
            print(f"   Snip  : ...{h['snippet']}...")
        if len(hits) > limit:
            print(f"\n(+{len(hits)-limit} more not shown)")

    print_hits("SINK EVIDENCE (dangerous write/execution points)", all_sinks)
    print_hits("SOURCE EVIDENCE (attacker-controlled inputs)", all_sources)

    print("\n" + "-"*78)
    print("MECHANISM #5 ASSESSMENT")
    print("-"*78)
    if len(all_sinks) == 0:
        print("✅ No obvious DOM-XSS sinks detected in scanned scripts.")
    else:
        print("⚠ DOM-XSS sinks detected. This is a potential risk.")
        print("   Next step: check if any sink uses untrusted sources without sanitization.")
        print("   (Static scan flags risk; it does not prove exploitability.)")

    return {
        "target_url": target_url,
        "final_url": final_url,
        "status": status,
        "sinks": all_sinks,
        "sources": all_sources,
        "score": score
    }

## Run all mechanisms

In [ ]:
import requests
from urllib.parse import urlparse

def build_session_for_target(target_url: str):
    """Create ONE session per target. If DVWA target and creds set → auto login & set security."""
    s = requests.Session()
    s.headers.update({"User-Agent": "Mozilla/5.0 (Integrated-Scanner)"})

    # DVWA detection: if target host matches DVWA_BASE host
    if DVWA_BASE and DVWA_USERNAME and DVWA_PASSWORD:
        try:
            dvwa_host = urlparse(DVWA_BASE).netloc
            tgt_host = urlparse(target_url).netloc
            if dvwa_host and (dvwa_host == tgt_host) and ("/dvwa" in urlparse(target_url).path or "dvwa" in target_url):
                # Use CSP notebook helper functions (dvwa_login + dvwa_set_security)
                s = dvwa_login(DVWA_BASE.rstrip('/') + '/login.php', DVWA_USERNAME, DVWA_PASSWORD)
                if DVWA_SECURITY_LEVEL:
                    dvwa_set_security(s, DVWA_BASE.rstrip('/') + '/security.php', DVWA_SECURITY_LEVEL)
                return s
        except Exception:
            pass

    return s


def is_public_site(url: str) -> bool:
    host = urlparse(url).netloc.lower()
    # crude heuristic: treat non-localhost/non-private IP as public
    return not (host.startswith("127.") or host.startswith("localhost") or host.startswith("192.168.") or host.startswith("10.") or host.startswith("172."))


def run_all_for_target(target_url: str):
    print("\n" + "#"*78)
    print(f"RUNNING ALL MECHANISMS FOR: {target_url}")
    print("#"*78)

    session = build_session_for_target(target_url)

    # ---------------------------
    # Mechanism 1: CSP
    # ---------------------------
    try:
        csp_res = check_content_security_policy(session, target_url)
        print_csp_results(csp_res)
    except Exception as e:
        print(f"\n[Mechanism 1 CSP] Error: {e}")

    # ---------------------------
    # Mechanism 2: Cookies/Session
    # ---------------------------
    try:
        _ = mech2_cookie_session_hardening(session, target_url)
    except Exception as e:
        print(f"\n[Mechanism 2 Cookies] Error: {e}")

    # ---------------------------
    # Mechanism 3: Security Headers
    # (uses its own request internally — we'll optimize later)
    # ---------------------------
    try:
        sh_res = mechanism3_security_headers_audit(target_url)
        print_mechanism3_report(sh_res)
    except Exception as e:
        print(f"\n[Mechanism 3 Headers] Error: {e}")

    # ---------------------------
    # Mechanism 4: Input Validation/Reflection
    # For PUBLIC sites, keep passive_only=True
    # ---------------------------
    try:
        passive_only = PASSIVE_ONLY_FOR_PUBLIC and is_public_site(target_url)
        m4 = run_mechanism_4(session=session, page_url=target_url, passive_only=passive_only)
        print_passive_validation(m4["passive"])
        if not passive_only:
            print_active_validation(m4["active"])
    except Exception as e:
        print(f"\n[Mechanism 4 Input Validation] Error: {e}")

    # ---------------------------
    # Mechanism 5: DOM sink/source scan
    # It accepts cookies dict, so we pass session cookies
    # ---------------------------
    try:
        cookies = session.cookies.get_dict()
        _ = mechanism5_dom_xss_scan(target_url, cookies=cookies)
    except Exception as e:
        print(f"\n[Mechanism 5 DOM XSS] Error: {e}")


# =========================
# RUN ALL TARGETS
# =========================
for t in TARGETS:
    run_all_for_target(t)
